# ACC Framework V2.2-FJE Analysis

This notebook analyzes the **single frozen Sahib-derived cost** campaign. It intentionally does not treat IAE/ISE/ITAE/ITSE as active V2 objectives.

Research guard: if the scenario manifest or cost profile is not frozen, calibration-dependent cells report that state instead of silently substituting development values.

In [ ]:
from pathlib import Path
import json, pickle, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from core.config import load_manifest, resolve_project_path
from core.checkpoint_jobs import load_all_job_checkpoints
from core.scenarios_v2 import load_scenario_suite
from benchmarks.acc_cost_fje_v2 import load_cost_profile, evaluate_performance_vector

MANIFEST = load_manifest()
assert MANIFEST["benchmark"] == "acc_cost_sahib_fje_v2"
BENCHMARK = MANIFEST["benchmark"]
ALGOS = list(MANIFEST["optimization"]["algorithms"])
OBJECTIVE = "proposed_cost_fje"
print("Benchmark:", BENCHMARK)
print("Algorithms:", ALGOS)
print("Expected main runs:", MANIFEST["optimization"]["total_jobs"])

In [ ]:
scenario_path = resolve_project_path(MANIFEST["cost_framework"]["scenario_manifest_path"])
profile_path = resolve_project_path(MANIFEST["cost_framework"]["cost_profile_path"])
scenario_meta = json.loads(scenario_path.read_text())
profile_meta = json.loads(profile_path.read_text())
print("Scenario status:", scenario_meta.get("status"))
print("Cost profile status:", profile_meta.get("status"))
if profile_meta.get("weights"):
    display(pd.DataFrame([profile_meta["weights"]]))

## Checkpoint summary

The final campaign contains one objective only. Runtime, FE count, and final fitness remain separate evidence dimensions.

In [ ]:
stats = load_all_job_checkpoints(BENCHMARK, ALGOS, [OBJECTIVE])
rows=[]
for algo in ALGOS:
    bucket=stats[OBJECTIVE][algo]
    for i, run_id in enumerate(bucket["run_ids"]):
        rows.append({
            "algorithm": algo,
            "run_id": run_id,
            "fitness": bucket["best_fitness"][i],
            "runtime_s": bucket["durations"][i],
            "function_evaluations": bucket["function_evaluations"][i],
            "solution": bucket["best_solutions"][i],
            "curve": bucket["curves"][i],
        })
runs = pd.DataFrame(rows)
print(f"Loaded {len(runs)} checkpoints")
runs.head()

In [ ]:
if len(runs):
    summary = runs.groupby("algorithm").agg(
        n=("fitness","count"),
        median=("fitness","median"),
        mean=("fitness","mean"),
        std=("fitness","std"),
        runtime_mean_s=("runtime_s","mean"),
        fe_mean=("function_evaluations","mean"),
    )
    q1=runs.groupby("algorithm")["fitness"].quantile(0.25)
    q3=runs.groupby("algorithm")["fitness"].quantile(0.75)
    summary["iqr"] = q3-q1
    display(summary.sort_values("median"))
else:
    print("No V2.2-FJE main checkpoints yet.")

## Median/IQR convergence

Each algorithm is summarized across repeated runs at each generation, matching the manuscript's median/IQR convergence interpretation.

In [ ]:
if len(runs):
    for algo in ALGOS:
        curves=[np.asarray(c,dtype=float) for c in runs.loc[runs.algorithm==algo,"curve"] if c is not None]
        if not curves: continue
        n=min(map(len,curves))
        arr=np.vstack([c[:n] for c in curves])
        x=np.arange(1,n+1)
        med=np.median(arr,axis=0)
        q1=np.quantile(arr,0.25,axis=0)
        q3=np.quantile(arr,0.75,axis=0)
        plt.plot(x,med,label=algo)
        plt.fill_between(x,q1,q3,alpha=.15)
    plt.xlabel("Iteration")
    plt.ylabel("Frozen V2.2-FJE scalar cost")
    plt.legend()
    plt.grid(alpha=.25)
    plt.show()

## Median-representative observed controller

For downstream validation, select the **observed run nearest each algorithm's median fitness**, not a synthetic median gain vector.

In [ ]:
representatives=[]
if len(runs):
    for algo,g in runs.groupby("algorithm"):
        med=float(g.fitness.median())
        idx=(g.fitness-med).abs().idxmin()
        r=g.loc[idx]
        representatives.append({"algorithm":algo,"run_id":int(r.run_id),"fitness":float(r.fitness),"solution":r.solution})
reps=pd.DataFrame(representatives)
display(reps)

## Re-evaluate physical components

This cell only runs when the cost profile and scenario manifest are frozen. It exposes the four underlying metrics so a smaller scalar cost is never interpreted without its physical components.

In [ ]:
if profile_meta.get("status") == "frozen" and scenario_meta.get("status") == "frozen" and len(reps):
    suite=load_scenario_suite(development=False, require_frozen=True)
    profile=load_cost_profile(require_frozen=True)
    physical=[]
    for _,r in reps.iterrows():
        perf=evaluate_performance_vector(r.solution,suite)
        physical.append({
            "algorithm":r.algorithm,
            "cruise_speed_rmse":perf.cruise_speed_rmse,
            "following_spacing_rmse":perf.following_spacing_rmse,
            "realized_jerk_rms":perf.realized_jerk_rms,
            "positive_tractive_energy_wh_per_km":perf.positive_tractive_energy_wh_per_km,
            "feasible":perf.feasible,
        })
    display(pd.DataFrame(physical))
else:
    print("Calibration artifacts are not frozen yet; physical-component audit is intentionally deferred.")

## Validation hooks still outside the fitness

After the external-file audit, validation should remain separate from cost construction:

- cruise rise time, settling time, overshoot, steady-state error;
- Casale-style Time to Equilibrium (TTE) for controlled following disturbances;
- TTC/minimum gap as continuous safety diagnostics;
- held-out ZalaZone/Casale/OpenACC episodes;
- WLTC moving segments;
- L3Pilot external validation;
- physical-parameter and sensing robustness.

Do not change `scenario_manifest_acc_cost_v2.json` after viewing research-calibration results without creating a new framework/profile version.